In [8]:
%pip install -U pip setuptools wheel
%pip install "datasets<4.0"
%pip install seqeval --no-build-isolation

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [9]:
%pip install transformers seqeval "datasets<4.0" pyarrow


Note: you may need to restart the kernel to use updated packages.


In [10]:
from collections import Counter
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer)
import pandas as pd
import numpy as np

# Load Dataset

In [11]:
# Dataset dimuat dari folder lokal nlp-terma/data (bukan dari HuggingFace Hub)
import os

DATA_CANDIDATES = ["../data", "nlp-terma/data", "data", "/content/nlp-terma/data", "/content/data"]
DATA_DIR = next((d for d in DATA_CANDIDATES if os.path.exists(os.path.join(d, "train.parquet"))), None)
assert DATA_DIR is not None, "train.parquet tidak ditemukan. Sesuaikan DATA_DIR ke folder nlp-terma/data."
print("DATA_DIR:", os.path.abspath(DATA_DIR))

# Urutan label sama persis dengan fitur IndoNLU TermA agar sel-sel EDA di bawah tetap valid
label_names = ["I-SENTIMENT", "O", "I-ASPECT", "B-SENTIMENT", "B-ASPECT"]
_L2ID = {n: i for i, n in enumerate(label_names)}

def load_split(name):
    df = pd.read_parquet(os.path.join(DATA_DIR, f"{name}.parquet"))
    df["tokens"] = df["tokens"].apply(lambda t: np.asarray(list(t), dtype=object))
    # label di file lokal berupa string -> ubah ke id integer
    df["seq_label"] = df["seq_label"].apply(lambda s: [_L2ID[x] for x in s])
    return df[["tokens", "seq_label"]]

train_ds = load_split("train")
val_ds = load_split("valid")
test_ds = load_split("test")

print(train_ds.info())
print("\n")

print(val_ds.info())
print("\n")

print(test_ds.info())
print("\n")

# --- Kode asli (HuggingFace Hub), dipertahankan sebagai referensi ---
# raw_ds = load_dataset("indonlp/indonlu", "terma", trust_remote_code=True)
#
# train_ds = raw_ds["train"].to_pandas()
#
# val_ds = raw_ds["validation"].to_pandas()
#
# test_ds = raw_ds["test"].to_pandas()
#
# print(train_ds.info())
# print("\n")
#
# print(val_ds.info())
# print("\n")
#
# print(test_ds.info())
# print("\n")

DATA_DIR: c:\Users\USER\Downloads\MLFT-dan-NLP-Practice\nlp-terma\data
<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   tokens     3000 non-null   object
 1   seq_label  3000 non-null   object
dtypes: object(2)
memory usage: 47.0+ KB
None


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   tokens     1000 non-null   object
 1   seq_label  1000 non-null   object
dtypes: object(2)
memory usage: 15.8+ KB
None


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   tokens     1000 non-null   object
 1   seq_label  1000 non-null   object
dtypes: object(2)
memory usage: 15.8+ KB
None




In [12]:
display(train_ds.head(10))
display("\n")

display(val_ds.head(10))
display("\n")

display(test_ds.head(10))
display("\n")

,tokens,seq_label
0,"[kamar, saya, ada, kendala, di, ac, tidak, ber...","[1, 1, 1, 1, 1, 4, 3, 0, 0, 1, 1, 1, 4, 2, 3, ..."
1,"[tempatnya, bagus, ., kolam, renangnya, bersih...","[4, 3, 1, 4, 2, 3, 1]"
2,"[oke, banget, ,, tetapi, ac, nya, tidak, bisa,...","[3, 0, 1, 1, 4, 2, 3, 0, 0, 1, 1, 1]"
3,"[pintu, geser, kurang, rapat, .]","[4, 2, 3, 0, 1]"
4,"[keren, ., nyaman, semuanya, .]","[3, 1, 3, 4, 1]"
5,"[tidak, dapat, snack, ., setelah, di, keluhan,...","[3, 0, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
6,"[pelayanan, lumayan, baik, .]","[4, 3, 0, 1]"
7,"[overall, bagus, ., terjangkau, ., kapan, kapa...","[4, 3, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
8,"[pelayanan, mas, nya, sangat, baik, ,, membant...","[4, 2, 2, 3, 0, 1, 3, 1, 3, 1, 3, 1, 4, 3, 0, ..."
9,"[airyroom, ini, sangat, mengecewakan, dalam, h...","[1, 1, 3, 0, 1, 1, 4, 1, 4, 2, 1]"


'\n'

,tokens,seq_label
0,"[over, all, enak, sih, ,, kalau, buat, young, ...","[4, 2, 3, 1, 1, 1, 1, 1, 1, 1, 1]"
1,"[bed, dan, ac, nya, nyaman, ., kamar, luas, .,...","[4, 1, 4, 2, 3, 1, 4, 3, 1, 1, 4, 2, 2, 3, 0, ..."
2,"[paling, nyaman, dari, yang, prnah, saya, inap...","[1, 3, 1, 1, 1, 1, 1, 1]"
3,"[meskipun, luas, kamarnya, pas, ,, tetapi, ber...","[1, 3, 4, 3, 1, 1, 3, 0, 1, 3, 4, 2, 1, 1, 1, ..."
4,"[pelayanan, baik, ., terimakasih, .]","[4, 3, 1, 1, 1]"
5,"[kamarnya, lumayan, bagus, ,, cuma, sayang, ai...","[4, 3, 0, 1, 1, 1, 4, 2, 3, 1, 1, 1, 3, 0, 1]"
6,"[(, +, ), kamar, luas, ,, ada, kolam, renang, ...","[1, 1, 1, 4, 3, 1, 3, 4, 2, 1, 1, 1, 1, 1, 1, ..."
7,"[tempat, nya, nyaman, fasilitas, oke, semua, .]","[4, 2, 3, 4, 3, 1, 1]"
8,"[kamar, banyak, nyamuk, ,, ac, tidak, dingin, ...","[4, 3, 0, 1, 4, 3, 0, 1, 1, 1, 1, 1, 1, 1, 1, ..."
9,"[air, panas, nya, tidak, banget, panas, ,, say...","[4, 2, 2, 3, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


'\n'

,tokens,seq_label
0,"[pelayanan, ramah, ,, kamar, nyaman, dan, fasi...","[4, 3, 1, 4, 3, 1, 4, 3, 1, 1, 4, 2, 3, 0, 1]"
1,"[tidak, terlalu, jauh, dari, pusat, kota, .]","[1, 1, 1, 1, 1, 1, 1]"
2,"[dengan, harga, terjangkau, kita, sudah, menda...","[1, 4, 3, 1, 1, 1, 4, 1, 3, 1]"
3,"[kondisinya, cukup, baik, ,, hanya, dapatnya, ...","[4, 3, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,"[kamar, bersih, ,, bentuknya, unik, ,, tetapi,...","[4, 3, 1, 4, 3, 1, 1, 4, 2, 3, 0, 1]"
5,"[room, bersih, tetapi, sayang, linen, kotor, ,...","[4, 3, 1, 1, 4, 3, 1, 3, 0, 0, 1, 4, 2, 3, 1, ..."
6,"[air, hangat, di, bathroom, tidak, berfungsi, ...","[4, 2, 1, 1, 3, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
7,"[hotelnya, seperti, kurang, terawat, ,, baik, ...","[4, 1, 3, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
8,"[baik, ,, tetapi, sudah, tidak, diadakan, di, ...","[3, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
9,"[bantal, keras, dan, tebal, ,, gorden, tidak, ...","[4, 3, 1, 3, 1, 4, 3, 0, 0, 0, 1, 3, 0, 4, 2, ..."


'\n'

# EDA

In [13]:
# Analisa jumlah token per komentar

def add_text_stats(df, tokens_col='tokens', tok_col='n_tok', sent_col='n_sent',
                   sent_enders={".", "!", "?"}, percentiles=[.5, .95, .99]):

    df[tok_col] = df[tokens_col].str.len()
    df[sent_col] = df[tokens_col].apply(
        lambda t: max(1, sum(w in sent_enders for w in t))
    )
    print(df[[tok_col, sent_col]].describe(percentiles=percentiles))
    print("\n")

    return df

train_ds = add_text_stats(train_ds)
val_ds = add_text_stats(val_ds)
test_ds = add_text_stats(test_ds)

             n_tok       n_sent
count  3000.000000  3000.000000
mean     15.911333     1.709333
std      12.482043     1.202218
min       2.000000     1.000000
50%      12.000000     1.000000
95%      42.000000     4.000000
99%      64.000000     7.000000
max      96.000000    11.000000


           n_tok       n_sent
count  1000.0000  1000.000000
mean     16.1240     1.710000
std      12.6049     1.196383
min       2.0000     1.000000
50%      12.0000     1.000000
95%      43.0000     4.000000
99%      62.0000     7.000000
max      96.0000     8.000000


             n_tok       n_sent
count  1000.000000  1000.000000
mean     15.498000     1.688000
std      11.600024     1.168769
min       3.000000     1.000000
50%      12.000000     1.000000
95%      39.050000     4.000000
99%      55.020000     6.010000
max      74.000000    11.000000




In [14]:
# label_names sudah didefinisikan pada sel load dataset (data lokal)
ID2L = {i: n for i, n in enumerate(label_names)}
L2ID = {n: i for i, n in ID2L.items()}

# --- Kode asli, dipertahankan sebagai referensi ---
# label_names = raw_ds["train"].features["seq_label"].feature.names
#
# ID2L = {i: n for i, n in enumerate(label_names)}
# L2ID = {n: i for i, n in ID2L.items()}

In [15]:
ID2L

{0: 'I-SENTIMENT', 1: 'O', 2: 'I-ASPECT', 3: 'B-SENTIMENT', 4: 'B-ASPECT'}

In [16]:
# Frekuensi Label

def print_label_distribution(df, id2label, label_col='seq_label', sort_by='id'):

    lab = Counter(l for s in df[label_col] for l in s)

    if sort_by == 'id':
        items = sorted(lab.items())
    elif sort_by == 'count':
        items = sorted(lab.items(), key=lambda kv: kv[1], reverse=True)
    elif sort_by == 'label':
        items = sorted(lab.items(), key=lambda kv: id2label[kv[0]])
    else:
        raise ValueError(f"Unknown sort_by: {sort_by!r}")

    result = {id2label[k]: v for k, v in items}

    print(result)
    print("\n")

print_label_distribution(train_ds, ID2L, sort_by='id')
print_label_distribution(val_ds, ID2L, sort_by='id')
print_label_distribution(test_ds, ID2L, sort_by='id')

{'I-SENTIMENT': 3249, 'O': 30205, 'I-ASPECT': 1716, 'B-SENTIMENT': 7264, 'B-ASPECT': 5300}


{'I-SENTIMENT': 1127, 'O': 10273, 'I-ASPECT': 569, 'B-SENTIMENT': 2414, 'B-ASPECT': 1741}


{'I-SENTIMENT': 1066, 'O': 9706, 'I-ASPECT': 584, 'B-SENTIMENT': 2384, 'B-ASPECT': 1758}




In [17]:
# span extraction

def spans(seq, b, i):
    seq = list(seq) + [1]          # paksa list, sentinel O di akhir
    out, start = [], None
    for k, l in enumerate(seq):
        if l == b:                 # B- selalu menutup span lama, membuka yang baru
            if start is not None:
                out.append((start, k))
            start = k
        elif l == i and start is not None:
            pass                   # I- melanjutkan span
        else:                      # O atau tag tipe lain: tutup span
            if start is not None:
                out.append((start, k))
            start = None
    return out

train_ds["asp"] = train_ds.seq_label.apply(lambda s: spans(s, 4, 2))
train_ds["sen"] = train_ds.seq_label.apply(lambda s: spans(s, 3, 0))
print("span ASPECT:", train_ds.asp.str.len().describe())
print("span SENTIMENT:", train_ds.sen.str.len().describe())
print("Persentase review tanpa aspek:", (train_ds.asp.str.len() == 0).mean())
print("Persentase review tanpa sentimen:", (train_ds.sen.str.len() == 0).mean())

total_aspects = sum(len(sp) for sp in train_ds.asp)
print("Total aspect mentions:", total_aspects)
print("len ASP:", Counter(e - s for sp in train_ds.asp for s, e in sp))

total_sents = sum(len(sp) for sp in train_ds.sen)
print("Total sent mentions:", total_sents)
print("len SENT:", Counter(e - s for sp in train_ds.sen for s, e in sp))

span ASPECT: count    3000.000000
mean        1.766667
std         1.477707
min         0.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        11.000000
Name: asp, dtype: float64
span SENTIMENT: count    3000.000000
mean        2.421333
std         1.628710
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        12.000000
Name: sen, dtype: float64
Persentase review tanpa aspek: 0.17233333333333334
Persentase review tanpa sentimen: 0.06566666666666666
Total aspect mentions: 5300
len ASP: Counter({1: 3858, 2: 1210, 3: 199, 4: 29, 5: 3, 6: 1})
Total sent mentions: 7264
len SENT: Counter({1: 4524, 2: 2325, 3: 335, 4: 71, 5: 8, 6: 1})


In [18]:
# Illegal structure
# Cek apakah ada i-sentiment berdiri tanpa b-sentiment (bgeitu pula dengan aspect)

def orphan_idx(seq, b, i):
    """Indeks token I- yang tidak didahului B-/I- tipe yang sama."""
    seq = list(seq)
    return [k for k, l in enumerate(seq)
            if l == i and (k == 0 or seq[k-1] not in (b, i))]

for name, b, i, col in [("asp", 4, 2, "inv_asp"), ("sen", 3, 0, "inv_sen")]:
    train_ds[col] = train_ds.seq_label.apply(lambda s: orphan_idx(s, b, i))

n = len(train_ds)
for name, col in [("asp", "inv_asp"), ("sen", "inv_sen")]:
    has = train_ds[col].apply(len) > 0
    print(f"{name}: {has.sum()} / {n} baris invalid ({has.mean():.2%}), "
          f"total token I- yatim: {train_ds[col].apply(len).sum()}")

print(train_ds[train_ds.inv_asp.apply(len) > 0].head(10)[["seq_label", "tokens", "inv_asp"]])

asp: 4 / 3000 baris invalid (0.13%), total token I- yatim: 4
sen: 4 / 3000 baris invalid (0.13%), total token I- yatim: 4
                                              seq_label  \
529   [4, 2, 2, 3, 4, 3, 1, 3, 4, 3, 4, 3, 1, 1, 1, ...   
808          [4, 3, 1, 4, 3, 0, 1, 1, 1, 1, 1, 2, 1, 1]   
1737  [1, 3, 0, 1, 4, 3, 2, 3, 1, 4, 1, 1, 3, 1, 4, ...   
1966   [4, 1, 3, 2, 3, 1, 4, 1, 1, 3, 1, 4, 2, 3, 0, 1]   

                                                 tokens inv_asp  
529   [check, in, out, cepat, ac, dingin, kamu, bers...    [22]  
808   [kamar, oke, ,, toilet, kurang, bersih, wc, ny...    [11]  
1737  [alhamdulillah, cukup, puas, ,, roof, top, nya...     [6]  
1966  [bantal, sama, tempat, tidurnya, basah, ,, lan...     [3]  


In [19]:
# Negasi / Intensifier

NEG = {"tidak", "bukan", "belum", "kurang", "tanpa", "nggak", "gak", "ga", "tak"}
INT = {"sangat", "banget", "sekali", "cukup", "lumayan", "agak", "terlalu"}

def first_tok(t, sp): return t[sp[0]]
def prev_tok(t, sp): return t[sp[0]-1] if sp[0] > 0 else None
def has(tokens, spans_, lex, where="inside"):
    c = 0
    for s, e in spans_:
        if where == "inside" and any(w in lex for w in tokens[s:e]): c += 1
        if where == "before" and s > 0 and tokens[s-1] in lex: c += 1
        if where == "after" and e < len(tokens) and tokens[e] in lex: c += 1
    return c
for name, lex in [("NEG", NEG), ("INT", INT)]:
    for w in ("inside", "before", "after"):
        n = sum(has(t, sp, lex, w) for t, sp in zip(train_ds.tokens, train_ds.sen))
        print(name, w, n)

NEG inside 1299
NEG before 4
NEG after 36
INT inside 1123
INT before 12
INT after 19


In [20]:
def find_example(tokens_list, spans_list, lex, where):
    for i, (toks, spans) in enumerate(zip(tokens_list, spans_list)):
        for s, e in spans:
            if where == "inside":
                hits = [w for w in toks[s:e] if w in lex]
                if hits:
                    return i, toks, (s, e), hits[0]
            elif where == "before":
                if s > 0 and toks[s - 1] in lex:
                    return i, toks, (s, e), toks[s - 1]
            elif where == "after":
                if e < len(toks) and toks[e] in lex:
                    return i, toks, (s, e), toks[e]
    return None


def show_row(row_idx, sent_span, matched, where):
    toks = train_ds.tokens.iloc[row_idx]
    labs = train_ds.seq_label.iloc[row_idx]
    asp_spans = train_ds.asp.iloc[row_idx]
    s, e = sent_span

    print(f"row      : {row_idx}")
    print(f"tokens   : {toks}")
    print(f"asp spans: {asp_spans}")
    print(f"sent span: {sent_span}  -> {toks[s:e]}")
    print(f"matched  : {matched}  (where={where})")
    print()

    # Per-token table
    print(f"{'idx':>4}  {'token':<15} {'label':<10} {'in_asp':<7} {'in_sent':<8} {'match':<6}")
    print("-" * 60)
    for i, (w, lab) in enumerate(zip(toks, labs)):
        in_asp = any(a_s <= i < a_e for a_s, a_e in asp_spans)
        in_sent = s <= i < e
        is_match = (w == matched)
        lab_name = ID2L.get(lab, f"?{lab}")
        print(f"{i:>4}  {w:<15} {lab_name:<10} {str(in_asp):<7} {str(in_sent):<8} {str(is_match):<6}")

    # Inline rendering
    print()
    line_tok, line_lab, line_mark = [], [], []
    for i, (w, lab) in enumerate(zip(toks, labs)):
        is_sent = s <= i < e
        is_match = (w == matched)
        tok_str = f"**{w}**" if is_match else w
        if is_sent:
            tok_str = f"[{tok_str}]"
        line_tok.append(tok_str)
        line_lab.append(ID2L.get(lab, "?"))
        line_mark.append("^" if is_match else ("S" if is_sent else " "))
    print(" ".join(line_tok))
    print(" ".join(f"{l:<{max(len(t),3)}}" for l, t in zip(line_lab, line_tok)))
    print(" ".join(f"{m:<{max(len(t),3)}}" for m, t in zip(line_mark, line_tok)))


for name, lex in [("NEG", NEG), ("INT", INT)]:
    for where in ("inside", "before", "after"):
        ex = find_example(train_ds.tokens, train_ds.sen, lex, where)
        print(f"\n{'=' * 60}\n=== {name} / {where} ===\n{'=' * 60}")
        if ex is None:
            print("(no example found)")
            continue
        row_idx, toks, sent_span, matched = ex
        show_row(row_idx, sent_span, matched, where)


=== NEG / inside ===
row      : 0
tokens   : ['kamar' 'saya' 'ada' 'kendala' 'di' 'ac' 'tidak' 'berfungsi' 'optimal'
 '.' 'dan' 'juga' 'wifi' 'koneksi' 'kurang' 'stabil' '.']
asp spans: [(5, 6), (12, 14)]
sent span: (6, 9)  -> ['tidak' 'berfungsi' 'optimal']
matched  : tidak  (where=inside)

 idx  token           label      in_asp  in_sent  match 
------------------------------------------------------------
   0  kamar           O          False   False    False 
   1  saya            O          False   False    False 
   2  ada             O          False   False    False 
   3  kendala         O          False   False    False 
   4  di              O          False   False    False 
   5  ac              B-ASPECT   True    False    False 
   6  tidak           B-SENTIMENT False   True     True  
   7  berfungsi       I-SENTIMENT False   True     False 
   8  optimal         I-SENTIMENT False   True     False 
   9  .               O          False   False    False 
  10  dan      

In [21]:
# def sample_10_percent(df, random_state=None, filename='sampled_10_percent.csv'):
#     sampled_df =  df.sample(frac=0.1, random_state=random_state)
#     sampled_df["seq_label_decoded"] = sampled_df["seq_label"].apply(lambda ids: [ID2L[i] for i in ids])
#     sampled_df.to_csv(filename, index=False)

# sample_10_percent(train_ds, 67, 'train_sample.csv')
# sample_10_percent(val_ds, 67, 'val_sample.csv')
# sample_10_percent(test_ds, 67, 'test_sample.csv')

In [22]:
MODEL_NAME = 'indobenchmark/indobert-base-p1'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokens_list = [list(t) for t in train_ds["tokens"].to_list()]
lengths = [
    len(tokenizer(toks, is_split_into_words=True, add_special_tokens=True)['input_ids'])
    for toks in tokens_list
]
lengths = np.array(lengths)
print("Jumlah sampel :", len(lengths))
print("Min           :", lengths.min())
print("Max           :", lengths.max())
print("Mean          :", lengths.mean())
print("Median (50)   :", np.percentile(lengths, 50))
print("Persentil 90  :", np.percentile(lengths, 90))
print("Persentil 95  :", np.percentile(lengths, 95))
print("Persentil 99  :", np.percentile(lengths, 99))
p95 = int(np.percentile(lengths, 95))
p99 = int(np.percentile(lengths, 99))

MAX_LEN = p99 + 1  # atau p95 kalau mau lebih agresif truncate
MAX_LEN = min(MAX_LEN, tokenizer.model_max_length)  # cap 512 buat BERT

print("Pakai max_length =", MAX_LEN)

truncated = (lengths > MAX_LEN).sum()
print(f"{truncated} sampel ({100*truncated/len(lengths):.2f}%) akan ter-truncate")

c:\Users\USER\.conda\envs\monk\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--indobenchmark--indobert-base-p1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Jumlah sampel : 3000
Min           : 4
Max           : 106
Mean          : 19.082
Median (50)   : 15.0
Persentil 90  : 36.0
Persentil 95  : 48.0
Persentil 99  : 71.0
Pakai max_length = 72
22 sampel (0.73%) akan ter-truncate


---
# EDA Lanjutan (Tambahan Insight)

> Bagian di bawah ini **ditambahkan** setelah EDA awal; tidak ada sel sebelumnya yang diubah atau dihapus.
> Tujuannya menjawab pertanyaan: *"Seberapa sulit masalah ekstraksi ASPECT & SENTIMENT ini, dan keputusan pemodelan apa yang dipengaruhi?"*
> Semua sel memakai variabel dari sel sebelumnya (`train_ds`, `val_ds`, `test_ds`, `ID2L`, `tokenizer`) sehingga harus dijalankan berurutan.

| No | Analisis | Keputusan yang dipengaruhi |
|---|---|---|
| A | Proporsi label & konsistensi split | Metrik (span F1), perlunya class weight, validitas split |
| B | Panjang span & kata paling sering | Fitur CRF (window), kesulitan boundary |
| C | OOV & ambiguitas kata | CRF (fitur leksikal) vs Transformer (subword) |
| D | Baseline lookup (majority tag) | Batas bawah: seberapa jauh model harus melampaui hafalan kata |
| E | Struktur ASPECT-SENTIMENT | Fitur konteks, jarak pasangan aspek-sentimen |
| F | Noise ortografi | Preprocessing (jangan normalisasi berlebihan) |
| G | Fertility subword IndoBERT | Alignment label ke subword, strategi label first-subword |
| H | Duplikasi & kebocoran antar split | Validitas evaluasi |
| I | Kepadatan entitas vs panjang kalimat | max_length & padding |

## A. Proporsi Label & Konsistensi Split
Frekuensi mentah (sel sebelumnya) sulit dibandingkan antar split karena ukuran berbeda. Di sini dinormalkan ke persen,
lalu dihitung rata-rata panjang span (`(B+I)/B`) dan rasio token entitas terhadap `O`.

In [23]:
def label_prop(df):
    c = Counter(l for s in df.seq_label for l in s)
    tot = sum(c.values())
    return pd.Series({ID2L[k]: v / tot for k, v in c.items()})

prop = pd.DataFrame({"train": label_prop(train_ds),
                     "val": label_prop(val_ds),
                     "test": label_prop(test_ds)}).sort_index()
display((prop * 100).round(2))

print("Selisih maksimum antar split per label (poin persen):")
print(((prop.max(axis=1) - prop.min(axis=1)) * 100).round(2).to_dict())

def span_stats(df, b_name, i_name):
    c = Counter(l for s in df.seq_label for l in s)
    nb_, ni_ = c[L2ID[b_name]], c[L2ID[i_name]]
    return nb_, ni_, (nb_ + ni_) / max(nb_, 1)

print()
for name, df in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    a = span_stats(df, "B-ASPECT", "I-ASPECT")
    s = span_stats(df, "B-SENTIMENT", "I-SENTIMENT")
    tot_tok = sum(len(x) for x in df.seq_label)
    ent_ratio = 1 - prop.loc["O", name]
    print(f"{name:5s} | rata2 panjang span ASPECT={a[2]:.2f} tok, SENTIMENT={s[2]:.2f} tok "
          f"| token entitas={ent_ratio:.1%} | O:entitas = {prop.loc['O', name] / ent_ratio:.2f} : 1")

,train,val,test
B-ASPECT,11.10,10.80,11.34
B-SENTIMENT,15.22,14.97,15.38
I-ASPECT,3.59,3.53,3.77
I-SENTIMENT,6.81,6.99,6.88
O,63.28,63.71,62.63


Selisih maksimum antar split per label (poin persen):
{'B-ASPECT': 0.55, 'B-SENTIMENT': 0.41, 'I-ASPECT': 0.24, 'I-SENTIMENT': 0.18, 'O': 1.09}

train | rata2 panjang span ASPECT=1.32 tok, SENTIMENT=1.45 tok | token entitas=36.7% | O:entitas = 1.72 : 1
val   | rata2 panjang span ASPECT=1.33 tok, SENTIMENT=1.47 tok | token entitas=36.3% | O:entitas = 1.76 : 1
test  | rata2 panjang span ASPECT=1.33 tok, SENTIMENT=1.45 tok | token entitas=37.4% | O:entitas = 1.68 : 1


**Cara membaca:** token `O` mendominasi (~63%), sehingga *token accuracy* akan tinggi walau entitas salah semua.
Karena itu evaluasi harus memakai **span-level F1 (seqeval)**, bukan accuracy. Selisih antar split yang kecil menandakan
split yang dipakai representatif (tidak ada *distribution shift* label).

## B. Panjang Span & Kata Paling Sering
Panjang span menentukan seberapa besar jendela konteks yang dibutuhkan; kata yang sama muncul sebagai ASPECT dan SENTIMENT menunjukkan ambiguitas peran.

In [24]:
for df in (val_ds, test_ds):
    df["asp"] = df.seq_label.apply(lambda s: spans(s, 4, 2))
    df["sen"] = df.seq_label.apply(lambda s: spans(s, 3, 0))

def span_texts(df, col):
    return [" ".join(t[s:e]) for t, sp in zip(df.tokens, df[col]) for s, e in sp]

for col, nm in [("asp", "ASPECT"), ("sen", "SENTIMENT")]:
    lens = Counter(e - s for sp in train_ds[col] for s, e in sp)
    tot = sum(lens.values())
    print(f"== Distribusi panjang span {nm} (train) ==")
    for k in sorted(lens)[:8]:
        print(f"  {k} token: {lens[k]:5d} ({lens[k]/tot:.1%})")
    print(f"  multi-token (>=2): {sum(v for k, v in lens.items() if k >= 2)/tot:.1%} | maks: {max(lens)} token")
    txt = span_texts(train_ds, col)
    cnt = Counter(txt)
    print(f"  total span={len(txt)}, unik={len(cnt)}, type/token ratio={len(cnt)/len(txt):.3f}")
    print("  top-15:", cnt.most_common(15))
    print()

asp_set = set(span_texts(train_ds, "asp"))
sen_set = set(span_texts(train_ds, "sen"))
both = asp_set & sen_set
print(f"Teks span yang muncul sebagai ASPECT *dan* SENTIMENT: {len(both)} "
      f"({len(both)/len(asp_set):.1%} dari tipe ASPECT unik)")
print("Contoh:", sorted(both)[:20])

== Distribusi panjang span ASPECT (train) ==
  1 token:  3858 (72.8%)
  2 token:  1210 (22.8%)
  3 token:   199 (3.8%)
  4 token:    29 (0.5%)
  5 token:     3 (0.1%)
  6 token:     1 (0.0%)
  multi-token (>=2): 27.2% | maks: 6 token
  total span=5300, unik=960, type/token ratio=0.181
  top-15: [('kamar', 547), ('pelayanan', 302), ('kamarnya', 194), ('ac', 178), ('kamar mandi', 159), ('wifi', 120), ('harga', 118), ('tempatnya', 117), ('lokasi', 100), ('fasilitas', 99), ('tempat', 98), ('air panas', 85), ('pelayanannya', 82), ('overall', 71), ('handuk', 66)]

== Distribusi panjang span SENTIMENT (train) ==
  1 token:  4524 (62.3%)
  2 token:  2325 (32.0%)
  3 token:   335 (4.6%)
  4 token:    71 (1.0%)
  5 token:     8 (0.1%)
  6 token:     1 (0.0%)
  multi-token (>=2): 37.7% | maks: 6 token
  total span=7264, unik=1432, type/token ratio=0.197
  top-15: [('bersih', 601), ('nyaman', 483), ('bagus', 287), ('ramah', 274), ('tidak ada', 246), ('baik', 218), ('oke', 215), ('kotor', 138), ('s

## C. OOV & Ambiguitas Kata
OOV tinggi pada token entitas = CRF berbasis leksikal akan kesulitan (butuh fitur sufiks/char n-gram); Transformer subword lebih tahan.
Kata dengan distribusi label campuran menunjukkan model **wajib memakai konteks**, tidak cukup lookup kata.

In [25]:
train_vocab = set(w for t in train_ds.tokens for w in t)

def oov_by_label(df):
    rows = []
    for toks, labs in zip(df.tokens, df.seq_label):
        for w, l in zip(toks, labs):
            rows.append((ID2L[l], w not in train_vocab))
    r = pd.DataFrame(rows, columns=["label", "oov"])
    return r.groupby("label").oov.agg(["mean", "sum", "count"]).rename(columns={"mean": "oov_rate"})

for name, df in [("val", val_ds), ("test", test_ds)]:
    r = oov_by_label(df)
    overall = sum(r["sum"]) / sum(r["count"])
    print(f"== OOV rate {name} (terhadap vocab train) | keseluruhan = {overall:.2%} ==")
    display(r.assign(oov_rate=(r.oov_rate * 100).round(2)))

# ambiguitas: kata dengan frekuensi >= 5 dan entropi label tinggi
from math import log2
word_lab = {}
for toks, labs in zip(train_ds.tokens, train_ds.seq_label):
    for w, l in zip(toks, labs):
        word_lab.setdefault(w, Counter())[ID2L[l]] += 1

def entropy(c):
    n = sum(c.values())
    return -sum(v / n * log2(v / n) for v in c.values())

amb = pd.DataFrame([(w, sum(c.values()), entropy(c), dict(c.most_common(3)))
                    for w, c in word_lab.items() if sum(c.values()) >= 5],
                   columns=["word", "freq", "entropy_bit", "label_dist"])
print("Kata paling ambigu (freq>=5, entropi label tertinggi):")
display(amb.sort_values("entropy_bit", ascending=False).head(15).reset_index(drop=True))
print(f"Kata (freq>=5) dengan >1 label berbeda: {(amb.entropy_bit > 0).mean():.1%} dari {len(amb)} kata")

== OOV rate val (terhadap vocab train) | keseluruhan = 0.00% ==


,oov_rate,sum,count
label,,,
B-ASPECT,0.0,0,1741
B-SENTIMENT,0.0,0,2414
I-ASPECT,0.0,0,569
I-SENTIMENT,0.0,0,1127
O,0.0,0,10273


== OOV rate test (terhadap vocab train) | keseluruhan = 4.65% ==


,oov_rate,sum,count
label,,,
B-ASPECT,3.07,54,1758
B-SENTIMENT,2.10,50,2384
I-ASPECT,3.42,20,584
I-SENTIMENT,3.75,40,1066
O,5.73,556,9706


Kata paling ambigu (freq>=5, entropi label tertinggi):


,word,freq,entropy_bit,label_dist
0,nyman,5,1.921928,"{'B-SENTIMENT': 2, 'I-SENTIMENT': 1, 'O': 1}"
1,koneksi,5,1.921928,"{'B-ASPECT': 2, 'I-ASPECT': 1, 'O': 1}"
2,standar,30,1.662155,"{'O': 14, 'B-SENTIMENT': 9, 'I-SENTIMENT': 6}"
3,sopan,23,1.632106,"{'I-SENTIMENT': 10, 'B-SENTIMENT': 9, 'O': 3}"
4,besar,51,1.578291,"{'B-SENTIMENT': 19, 'O': 17, 'I-SENTIMENT': 15}"
5,sesuai,129,1.573864,"{'I-SENTIMENT': 50, 'O': 42, 'B-SENTIMENT': 37}"
6,bermasalah,11,1.572624,"{'B-SENTIMENT': 4, 'O': 4, 'I-SENTIMENT': 3}"
7,lainnya,18,1.571542,"{'O': 7, 'B-ASPECT': 6, 'I-ASPECT': 5}"
8,lama,58,1.563192,"{'O': 24, 'B-SENTIMENT': 18, 'I-SENTIMENT': 16}"
9,closet,7,1.556657,"{'B-ASPECT': 3, 'O': 2, 'I-ASPECT': 2}"


Kata (freq>=5) dengan >1 label berbeda: 63.6% dari 840 kata


## D. Baseline Lookup (Majority Tag per Kata)
Baseline paling sederhana: tiap kata diberi label mayoritasnya di train (kata asing -> `O`).
Hasilnya adalah **lantai performa**; model CRF/Transformer harus jauh di atas ini agar klaim "model memahami konteks" bermakna.

In [26]:
from seqeval.metrics import classification_report as seq_report, f1_score as seq_f1

majority = {w: ID2L_name.most_common(1)[0][0]
            for w, ID2L_name in word_lab.items()}

def predict_lookup(df):
    return [[majority.get(w, "O") for w in toks] for toks in df.tokens]

def gold_names(df):
    return [[ID2L[l] for l in s] for s in df.seq_label]

for name, df in [("val", val_ds), ("test", test_ds)]:
    y_true, y_pred = gold_names(df), predict_lookup(df)
    flat_t = [x for s in y_true for x in s]
    flat_p = [x for s in y_pred for x in s]
    tok_acc = np.mean([a == b for a, b in zip(flat_t, flat_p)])
    print(f"== Baseline lookup [{name}] | token accuracy={tok_acc:.3f} | micro span F1={seq_f1(y_true, y_pred):.4f} ==")
    print(seq_report(y_true, y_pred, digits=3))

== Baseline lookup [val] | token accuracy=0.852 | micro span F1=0.6412 ==
              precision    recall  f1-score   support

      ASPECT      0.615     0.829     0.706      1741
   SENTIMENT      0.523     0.687     0.594      2414

   micro avg      0.562     0.746     0.641      4155
   macro avg      0.569     0.758     0.650      4155
weighted avg      0.561     0.746     0.641      4155

== Baseline lookup [test] | token accuracy=0.830 | micro span F1=0.6193 ==
              precision    recall  f1-score   support

      ASPECT      0.596     0.770     0.672      1761
   SENTIMENT      0.521     0.653     0.580      2386

   micro avg      0.554     0.702     0.619      4147
   macro avg      0.559     0.711     0.626      4147
weighted avg      0.553     0.702     0.619      4147



**Interpretasi:** bandingkan *token accuracy* (tinggi karena dominasi `O`) dengan *span F1* (jauh lebih rendah/berbeda).
Selisih kedua angka ini adalah alasan utama memakai span F1 sebagai metrik utama.

## E. Struktur Hubungan ASPECT-SENTIMENT
Berapa jauh sebuah sentimen dari aspek terdekatnya, dan apakah sentimen biasanya muncul sebelum/sesudah aspek?
Ini menentukan lebar jendela fitur konteks CRF (mis. `-2..+2`) dan apakah attention global Transformer diperlukan.

In [27]:
def nearest_gap(asp_spans, sen_spans):
    out = []
    for a_s, a_e in asp_spans:
        best = None
        for s_s, s_e in sen_spans:
            gap = max(s_s - a_e, a_s - s_e, 0)
            before = s_e <= a_s
            if best is None or gap < best[0]:
                best = (gap, before)
        if best is not None:
            out.append(best)
    return out

pairs = [p for a, s in zip(train_ds.asp, train_ds.sen) for p in nearest_gap(a, s)]
gaps = np.array([g for g, _ in pairs])
before = np.array([b for _, b in pairs])
print(f"Jumlah aspek yang punya sentimen di review yang sama: {len(pairs)}")
print(f"Gap ke sentimen terdekat (token): median={np.median(gaps)}, p90={np.percentile(gaps, 90)}, "
      f"p95={np.percentile(gaps, 95)}, maks={gaps.max()}")
for k in (0, 1, 2, 3, 5):
    print(f"  gap <= {k}: {np.mean(gaps <= k):.1%}")
print(f"Sentimen terdekat berada SEBELUM aspek: {before.mean():.1%} | SESUDAH: {1 - before.mean():.1%}")

n_asp = train_ds.asp.str.len()
n_sen = train_ds.sen.str.len()
combo = pd.crosstab(n_asp.clip(upper=4), n_sen.clip(upper=4))
combo.index.name, combo.columns.name = "n_aspect (4=4+)", "n_sentiment (4=4+)"
print("\nCrosstab jumlah span ASPECT x SENTIMENT per review:")
display(combo)
print(f"Korelasi jumlah aspek vs jumlah sentimen per review: {np.corrcoef(n_asp, n_sen)[0, 1]:.3f}")
print(f"Review dengan sentimen tanpa aspek: {((n_sen > 0) & (n_asp == 0)).mean():.2%} | "
      f"aspek tanpa sentimen: {((n_asp > 0) & (n_sen == 0)).mean():.2%} | keduanya kosong: {((n_asp == 0) & (n_sen == 0)).mean():.2%}")

Jumlah aspek yang punya sentimen di review yang sama: 5299
Gap ke sentimen terdekat (token): median=0.0, p90=1.0, p95=2.0, maks=16
  gap <= 0: 79.6%
  gap <= 1: 92.2%
  gap <= 2: 96.1%
  gap <= 3: 98.3%
  gap <= 5: 99.6%
Sentimen terdekat berada SEBELUM aspek: 21.8% | SESUDAH: 78.2%

Crosstab jumlah span ASPECT x SENTIMENT per review:


n_sentiment (4=4+),0,1,2,3,4
n_aspect (4=4+),,,,,
0,196,159,112,40,10
1,1,520,315,123,43
2,0,33,405,221,109
3,0,7,30,207,131
4,0,2,8,27,301


Korelasi jumlah aspek vs jumlah sentimen per review: 0.800
Review dengan sentimen tanpa aspek: 10.70% | aspek tanpa sentimen: 0.03% | keduanya kosong: 6.53%


## F. Noise Ortografi
Seberapa "kotor" teks review (angka, tanda baca, huruf berulang, token 1 huruf, kapital)? Menentukan seberapa jauh preprocessing boleh dilakukan.

In [28]:
import re

def ortho_stats(df):
    toks = [w for t in df.tokens for w in t]
    n = len(toks)
    return {
        "token_total": n,
        "has_digit": sum(any(c.isdigit() for c in w) for w in toks) / n,
        "punct_only": sum(not any(c.isalnum() for c in w) for w in toks) / n,
        "elongation (aaa)": sum(bool(re.search(r"(.)\1\1", w)) for w in toks) / n,
        "1-huruf": sum(len(w) == 1 and w.isalpha() for w in toks) / n,
        "ada_kapital": sum(any(c.isupper() for c in w) for w in toks) / n,
    }

display(pd.DataFrame({k: ortho_stats(v) for k, v in
                      [("train", train_ds), ("val", val_ds), ("test", test_ds)]}).round(4))

# apakah noise berasosiasi dengan label entitas?
rows = []
for toks, labs in zip(train_ds.tokens, train_ds.seq_label):
    for w, l in zip(toks, labs):
        rows.append((ID2L[l], any(c.isdigit() for c in w), not any(c.isalnum() for c in w), len(w)))
r = pd.DataFrame(rows, columns=["label", "digit", "punct", "len"])
print("Rata-rata per label (train):")
display(r.groupby("label").agg(digit=("digit", "mean"), punct=("punct", "mean"), mean_len=("len", "mean")).round(4))

,train,val,test
token_total,47734.0000,16124.0000,15498.0000
has_digit,0.0084,0.0092,0.0081
punct_only,0.1789,0.1777,0.1792
elongation (aaa),0.0002,0.0001,0.0001
1-huruf,0.0006,0.0006,0.0005
ada_kapital,0.0000,0.0000,0.0000


Rata-rata per label (train):


,digit,punct,mean_len
label,,,
B-ASPECT,0.0000,0.0000,6.4126
B-SENTIMENT,0.0001,0.0000,5.5202
I-ASPECT,0.0000,0.0076,4.9959
I-SENTIMENT,0.0049,0.0012,5.7458
O,0.0126,0.2821,3.9674


## G. Fertility Subword IndoBERT per Jenis Label
*Fertility* = jumlah subword per kata. Kata entitas yang terpecah banyak (typo, serapan) membuat alignment label BIO ke subword
harus dirancang hati-hati (label hanya di subword pertama, sisanya `-100`).

In [29]:
fert_rows = []
unk_id = tokenizer.unk_token_id
for toks in train_ds.tokens:
    enc = tokenizer(list(toks), is_split_into_words=True, add_special_tokens=False)
    wid = enc.word_ids()
    cnt = Counter(wid)
    unk = Counter(w for w, i in zip(wid, enc["input_ids"]) if i == unk_id)
    fert_rows.append([(cnt[i], unk.get(i, 0)) for i in range(len(toks))])

recs = []
for fr, labs in zip(fert_rows, train_ds.seq_label):
    for (f, u), l in zip(fr, labs):
        recs.append((ID2L[l], f, u))
fd = pd.DataFrame(recs, columns=["label", "fertility", "unk"])
fd["group"] = fd.label.str.split("-").str[-1]  # O / ASPECT / SENTIMENT
summary = fd.groupby("group").agg(mean_fertility=("fertility", "mean"),
                                   pct_split=("fertility", lambda x: (x > 1).mean()),
                                   pct_ge3=("fertility", lambda x: (x >= 3).mean()),
                                   pct_unk=("unk", lambda x: (x > 0).mean())).round(4)
display(summary)
print(f"Rasio total subword/kata (train): {fd.fertility.mean():.3f}")
print("Kata entitas dengan fertility>=4 (contoh yang paling terpecah):")
ex = []
for toks, fr, labs in zip(train_ds.tokens, fert_rows, train_ds.seq_label):
    for w, (f, u), l in zip(toks, fr, labs):
        if f >= 4 and ID2L[l] != "O":
            ex.append((w, ID2L[l], f))
print(Counter(ex).most_common(15))

,mean_fertility,pct_split,pct_ge3,pct_unk
group,,,,
ASPECT,1.1420,0.1253,0.0143,0.0
O,1.0631,0.0503,0.0108,0.0
SENTIMENT,1.0580,0.0459,0.0107,0.0


Rasio total subword/kata (train): 1.074
Kata entitas dengan fertility>=4 (contoh yang paling terpecah):
[(('complimentary', 'B-ASPECT', 4), 4), (('amenities', 'B-ASPECT', 4), 2), (('prncahayaan', 'B-ASPECT', 4), 1), (('kecoaknya', 'I-SENTIMENT', 4), 1), (('remangremang', 'B-SENTIMENT', 4), 1), (('kereception', 'B-ASPECT', 4), 1), (('reccomended', 'B-SENTIMENT', 4), 1), (('nyangkut', 'I-SENTIMENT', 4), 1), (('remangremang', 'I-SENTIMENT', 4), 1), (('perlengakapannya', 'B-ASPECT', 4), 1), (('aesthetic', 'B-SENTIMENT', 4), 1), (('smookingnya', 'I-ASPECT', 4), 1), (('rekomeded', 'B-SENTIMENT', 4), 1), (('selimut/sprey', 'B-ASPECT', 4), 1), (('enaakkmuraahh', 'B-SENTIMENT', 6), 1)]


## H. Duplikasi & Kebocoran Antar Split
Kalimat identik antar split membuat skor evaluasi optimistis (leakage). Duplikat dengan label berbeda menunjukkan inkonsistensi anotasi.

In [30]:
def key_series(df):
    return df.tokens.apply(lambda t: " ".join(t))

k_tr, k_va, k_te = key_series(train_ds), key_series(val_ds), key_series(test_ds)
print(f"Duplikat dalam split -> train: {k_tr.duplicated().sum()}, val: {k_va.duplicated().sum()}, test: {k_te.duplicated().sum()}")
print(f"Overlap kalimat -> trainâˆ©val: {len(set(k_tr) & set(k_va))}, trainâˆ©test: {len(set(k_tr) & set(k_te))}, valâˆ©test: {len(set(k_va) & set(k_te))}")

# duplikat dalam train dengan label berbeda
tmp = pd.DataFrame({"k": k_tr, "lab": train_ds.seq_label.apply(lambda s: tuple(s))})
conf = tmp.groupby("k").lab.nunique()
conf = conf[conf > 1]
print(f"Kalimat duplikat di train dengan anotasi BERBEDA: {len(conf)}")
for k in list(conf.index)[:3]:
    print("  ", k)

Duplikat dalam split -> train: 30, val: 4, test: 5
Overlap kalimat -> trainâˆ©val: 996, trainâˆ©test: 21, valâˆ©test: 6
Kalimat duplikat di train dengan anotasi BERBEDA: 1
   lokasi cukup strategis meskipun di jalan satu mobil . seprai set lebih ditingkatkan kebersihannya . kamar dan kamar mandi akan terasa lebih segar apabila diberi pengharum ruangan . snack ada 3 macam cukup mengenyangkan untuk sarapan . air minum mineral botol sebaiknya yang 600ml , jangan 330ml karena terlalu sedikit ( dari sisi harga juga tidak beda banyak ) .


## I. Kepadatan Entitas vs Panjang Kalimat
Apakah kalimat panjang lebih "jarang" entitas? Menunjukkan seberapa besar padding terbuang & apakah perlu *dynamic padding*.

In [31]:
d = train_ds.copy()
d["n_ent_tok"] = d.seq_label.apply(lambda s: sum(l != 1 for l in s))
d["ent_density"] = d.n_ent_tok / d.n_tok
d["len_bin"] = pd.cut(d.n_tok, bins=[0, 5, 10, 20, 40, 100], labels=["<=5", "6-10", "11-20", "21-40", ">40"])
tbl = d.groupby("len_bin", observed=True).agg(n_review=("n_tok", "size"),
                                              mean_ent_density=("ent_density", "mean"),
                                              mean_asp_span=("asp", lambda x: x.str.len().mean()),
                                              mean_sen_span=("sen", lambda x: x.str.len().mean()),
                                              pct_no_entity=("n_ent_tok", lambda x: (x == 0).mean()))
display(tbl.round(3))
print(f"Korelasi n_tok vs jumlah span (ASPECT+SENTIMENT): "
      f"{np.corrcoef(d.n_tok, d.asp.str.len() + d.sen.str.len())[0, 1]:.3f}")
mean_len = d.n_tok.mean()
print(f"Dengan padding statis ke panjang maksimum ({d.n_tok.max()}), efisiensi token ~ {mean_len / d.n_tok.max():.1%}; "
      f"dynamic padding sangat dianjurkan.")

,n_review,mean_ent_density,mean_asp_span,mean_sen_span,pct_no_entity
len_bin,,,,,
<=5,370,0.591,0.716,1.227,0.054
6-10,908,0.505,1.176,1.895,0.062
11-20,1014,0.404,1.783,2.410,0.081
21-40,541,0.326,2.732,3.384,0.063
>40,167,0.255,4.078,4.874,0.024


Korelasi n_tok vs jumlah span (ASPECT+SENTIMENT): 0.583
Dengan padding statis ke panjang maksimum (96), efisiensi token ~ 16.6%; dynamic padding sangat dianjurkan.


## Ringkasan Insight & Implikasi Pemodelan

Gabungan hasil EDA awal dan lanjutan (angka dari sel awal ditulis ulang untuk konteks):

| Temuan | Implikasi untuk model |
|---|---|
| Token `O` ~63% dan label ASPECT/SENTIMENT tidak seimbang; B-SENTIMENT paling banyak, I-ASPECT paling sedikit | Gunakan **span-level F1 (seqeval)** sebagai metrik utama; pantau per-tipe (ASPECT vs SENTIMENT) |
| 17,2% review tanpa aspek (dan sebagian tanpa sentimen) | Model harus mampu mengeluarkan "tidak ada entitas"; jangan paksa selalu prediksi span |
| Struktur BIO hampir bersih (hanya 4 baris / 0,13% `I-` yatim) | Cukup normalisasi ringan; tidak perlu rekonstruksi besar. Tetap terapkan validasi/perbaikan BIO saat dekode |
| Negasi (`tidak`, `kurang`) dan intensifier banyak berada **di dalam** span sentimen (NEG inside 1299, INT inside 1123), jarang di luar | Fitur konteks CRF harus menangkap frasa sentimen multi-kata; boundary error pada sentimen adalah risiko utama |
| Panjang token median 12, p99 64; subword IndoBERT p99 71 | `MAX_LEN=72` memangkas hanya 0,73% sampel; pakai **dynamic padding** agar efisien |
| OOV pada token entitas (sel C) dan fertility subword (sel G) | Jika OOV/fertility entitas tinggi -> tambah char n-gram/prefix-suffix pada CRF, Transformer relatif unggul |
| Baseline lookup (sel D) | Menjadi lantai performa; selisih ke CRF/Transformer = kontribusi konteks |
| Kata ambigu lintas label (sel C) | Konfirmasi bahwa konteks sekitar kata wajib dimodelkan |

**Catatan keterbatasan EDA:** hanya ~3.000 review train (hotel, satu domain); temuan tidak otomatis berlaku pada domain lain.
Daftar kata negasi/intensifier dibuat manual sehingga hitungannya adalah perkiraan bawah, bukan jumlah pasti.